## In-Class Lecture 23-1 - Optimizing the Size of C Structs

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass23
    cd inclass23

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

### Obtaining the class files

Perform the following command:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec23/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
Once you perform these commands, you will get the following:

    Makefile  mem_map.c  struct_align.c  struct_align_fix.c

We will learn how to optimize the size of a C struct when you design them in memory

## Optimal Size

In the code <code>struct_align.c</code>, we will calculate the expected size of the array. We will create a <code>size_t</code> variable - which is a compiler directive for <code>long unsigned int</code> - and add the sizes of the variables inside of the struct:

    typedef struct first_struct{
        float first_float;
        double first_double;
        char first_char;
        double second_double;
        int first_int;
        float second_float;
        char second_char;
        char third_char;
        float third_float;
    }first_struct;

So we have three <code>char</code>, three <code>float</code>, one <code>double</code>, and one <code>int</code>.

We will update struct_align.c to print this value.

## Compilation Run 

    make struct_align
    valgrind --leak-check=full ./struct_align
    
## Sample Run

    Size of optimal first_struct = 34
    Size of first_struct = 48
    The locations:
    Base address  : 0x4a4a040
    first_float   : 0x4a4a040
    first_double  : 0x4a4a048
    first_char    : 0x4a4a050
    second_double : 0x4a4a058
    first_int     : 0x4a4a060
    second_float  : 0x4a4a064
    second_char   : 0x4a4a068
    third_char    : 0x4a4a069
    third_float   : 0x4a4a06c
    
## What happened?

The Operating System lays out the data types as efficiently as possible. But 

    char - May be at any location since a char is a size of 1 byte
    int/float/unsigned int  - Must be at a location that is a multiple of 4 since they are a size of 4 bytes
    long int/double/pointer address  - Must be at a location that is a multiple of 8 since they are a size of 8 bytes
    
So let's break that down by memory location

    first_float   : 0x4a4a040 - Satisfies float rule and must be at the same location as the base address on the heap
    
    first_double  : 0x4a4a048 - We lose 4 bytes because a double can't be at 0x040 since 4 is not a multiple of 8. 
    
    first_char    : 0x4a4a050 - Goes to the first available spot at 0x050
    
    second_double : 0x4a4a058 - We lose 7 bytes because a double can't be at 0x051 since 1 is not a multiple of 8.
    
    first_int     : 0x4a4a060 - Goes to the first available spot at 0x060 and satisfies int rule
    
    second_float  : 0x4a4a064 - Goes to the first available spot at 0x064 and satisfies float rule
    
    second_char   : 0x4a4a068 - Goes to the first available spot at 0x068 and satisfies char rule
    
    second_char   : 0x4a4a069 - Goes to the first available spot at 0x069 and satisfies char rule
    
    third_float   : 0x4a4a06c - We lose 2 bytes because a double can't be at 0x06a since 10 is not a multiple of 4.

## So how can we improve this?

We arrange the data in the struct <b>in order of size from smallest to largest</b>.

We will go to <code>struct_align_fix.c</code> and re-write the struct.

When we run the program, what will be observe?